# Tanzania voice benchmark — Phase 3

**EXPERIMENTAL / untested.** Goal: hear candidate voice engines read the
`TANZANIA_VOICE_BENCHMARK.md` sentences so you can score them against the §4 bar.

Before you start:
1. `Runtime → Change runtime type → T4 GPU` (free tier is enough)
2. Have one clean **~10–20s mono `.wav`** of your normal speaking voice ready
3. `Runtime → Run all`, then upload the wav when asked

Outputs land in `out/xtts/` (your voice, Swahili unofficial) and `out/mms/`
(real Swahili, not your voice — the accent baseline). Listen to both, then fill
the tables in `TANZANIA_VOICE_BENCHMARK.md` / `MODEL_EVALUATION.md`.

In [ ]:
# 1. confirm the GPU
!nvidia-smi -L || print('NO GPU — set Runtime > Change runtime type > T4 GPU')

In [ ]:
# 2. get the benchmark tooling
!git clone --depth 1 -b phase-2-mock-skeleton https://github.com/nyumbafasta07-sketch/nyumbafasta-video-studio-1 repo 2>/dev/null || (cd repo && git pull)
%cd repo/worker/colab
!ls

In [ ]:
# 3. install backends (a few minutes)
!pip -q install coqui-tts "transformers>=4.44" torch soundfile

In [ ]:
# 4. upload your reference voice clip (.wav)
from google.colab import files
up = files.upload()
ref = next(iter(up))
print('using reference:', ref)

In [ ]:
# 5a. XTTS v2 — clones your voice. 'sw' is not official; judge the result.
!COQUI_TOS_AGREED=1 python voice_worker.py --mode benchmark --backend xtts --ref "$ref" --out ./out

In [ ]:
# 5b. MMS-TTS (swh) — real Swahili, fixed speaker. Accent/pronunciation baseline.
!python voice_worker.py --mode benchmark --backend mms --out ./out

In [ ]:
# 6. listen
import IPython.display as ipd, glob, json, os
for backend in ['xtts', 'mms']:
    print('\n==================', backend, '==================')
    res = f'out/{backend}/_results.json'
    if os.path.exists(res):
        for r in json.load(open(res)):
            print(f"[{r['id']}] {r['category']:14s} {r['status']}")
    for f in sorted(glob.glob(f'out/{backend}/*.wav')):
        print(f)
        ipd.display(ipd.Audio(f))

## Score it

For each backend, use the grid in `TANZANIA_VOICE_BENCHMARK.md`. Write down the
**specific** failure (accent drift? `ng'ombe` / `Mng'ong'o` wrong? rushed?
robotic? code-switch in cell D unnatural?). That decides the next move: a
different model, or a small fine-tune on your recordings (brief §8).

## Optional — run the app against the real voice

Run the cell below, copy the `https://…trycloudflare.com` URL, and on your
machine put it in `.env` as `GPU_PROVIDER=http` + `GPU_WORKER_URL=<that url>`.
Avatar / lip-sync / render stay mock until Phases 4–5.

In [ ]:
# 7. (optional) serve the voice worker over a public URL
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared
import subprocess, threading, time
threading.Thread(target=lambda: subprocess.run(
    ['python', 'voice_worker.py', '--mode', 'serve', '--backend', 'xtts', '--ref', ref, '--port', '8800']
), daemon=True).start()
time.sleep(5)
!cloudflared tunnel --url http://localhost:8800